In [1]:
from ipywidgets import (
    Dropdown, IntSlider, FloatSlider, Checkbox, HTML,
    HBox, VBox, Button, Output, IntText, Label, interactive_output
)
from scipy.signal import hilbert
from scipy.interpolate import CubicSpline, PchipInterpolator, Akima1DInterpolator, interp1d
from PyEMD import CEEMDAN, EMD, EEMD
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import mne


fs = 2000 # Samling freq
# Synthetic Signal 5Hz, 48Hz, 50Hz
freqs = [5, 48, 50]
amps = [1, 1, 1]
duration = 2
global t
t = np.arange(0, duration, 1/fs)

def make_signal_from_freq(duration, amps, freqs, t):
    signal = 0
    for i in range(0, len(freqs)):
        signal += amps[i]*np.sin(2*np.pi*freqs[i]*t)
    return signal

def add_noise_to_signal(signal, snr_db):
    sig_power = np.mean(signal**2)
    noise_power = sig_power / (10**(snr_db/10))
    noise = np.random.randn(*signal.shape) * np.sqrt(noise_power)
    return signal + noise

def add_offset_to_signal(signal, offset):
    return signal + offset

def plot_signal(t, signal, duration, title):
    plt.figure(figsize=(10,2.8))
    plt.plot(t, signal)
    plt.title(title)
    plt.xlim(0, duration)
    plt.show()


#sig = make_signal_from_freq(duration, amps, freqs, t)
#plot_signal(t, sig, duration)

#noisy_sig = add_noise_to_signal(sig, 10)
#plot_signal(t, noisy_sig, duration)

#offset_noisy_sig = add_offset_to_signal(noisy_sig, 0.5)
#plot_signal(t, offset_noisy_sig, duration)


In [2]:
# EMD Setup
def zero_crossings(x):
    return np.sum(x[:-1]*x[1:] < 0) # Detects when signal crosses zero

def is_imf(x):
    dx = np.diff(x)
    s = np.sign(np.where(dx == 0, 1e-12, dx))
    ds = np.diff(s)
    
    n_ext = np.sum(ds < 0) + np.sum(ds > 0) # Total number of extrema
    n_zero = zero_crossings(x) # Counte zero crossings
    
    return abs(n_ext - n_zero) <= 1 # If extrema and zero crossings differ by at most one

def sd_criterion(h_prev, h_cur): # Measure how much signal changes between sifts in EMD
    num = np.sum((h_prev - h_cur)**2)
    den = np.sum(h_prev**2) + 1e-16
    return num/den
    
def extend_signal(t, x, mode, n_ext=2): # Extends the signal through mirroring or using slopes
    if mode == "none" or len(x) < 3:
        return t, x

    dt = np.median(np.diff(t))

    if mode in {"mirror", "mirror_taper"}:
        n_ext = min(n_ext, len(x)-1) # dont extend m,ore points than available

        x_left = x[1:1+n_ext][::-1]
        x_right = x[-2-n_ext+1:-1][::-1] # mirror around first and last point

        t_left = t[0] - np.arange(n_ext, 0, -1)*dt
        t_right = t[-1] + np.arange(1, n_ext+1)*dt # new time coords for mirrored sections

        t_ext = np.concatenate([t_left, t, t_right])
        x_ext = np.concatenate([x_left, x, x_right]) # combine signal with mirrors

        if mode == "mirror_taper":
            L = len(x_left)
            R = len(x_right) # length of mirrors
            if L>0:
                wL = 0.5*(1-np.cos(np.linspace(0,np.pi, L)))
                x_ext[:L] = x_ext[:L]*(1-wL) # build cosine window wL from zero to pi to fade mirrored samples to zero

            if R>0:
                wR = 0.5*(1-np.cos(np.linspace(0,np.pi, R)))
                x_ext[-R:] = x_ext[-R:]*(1-wR) # same

        return t_ext, x_ext

    if mode == "linear": # extrapolate edges using local slopes
        l_slope = (x[1]-x[0])/(t[1]-t[0])
        r_slope = (x[-1]-x[-2])/(t[-1]-t[-2]) # Estimate slope at each end

        t_left = t[0] - np.arange(n_ext, 0, -1)*dt
        t_right = t[-1] + np.arange(1, n_ext+1)*dt # new time coords

        x_left = x[0] + l_slope*(t_left - t[0])
        x_right = x[-1] + r_slope*(t_right - t[-1]) # Compute extrapolated signal by extending with same slope

        return np.concatenate([t_left, t, t_right]), np.concatenate([x_left, x, x_right])

    return t, x

def make_envelopes(t, x, spline_type="natural", boundary="mirror"):
    dx = np.diff(x) # first derivative
    s = np.sign(np.where(dx == 0, 1e-12, dx)) # sign of slope
    ds = np.diff(s) # change in slope

    max_idx = np.where(ds < 0)[0] + 1 # when slope crosses the x-axis from + to -
    min_idx = np.where(ds > 0)[0] + 1 # Same but opposite

    extrema_max = np.unique(np.concatenate(([0], max_idx, [len(x)-1])))
    extrema_min = np.unique(np.concatenate(([0], min_idx, [len(x)-1]))) # Ensure first and last samples are candidates for upper/lower envelope

    if extrema_max.size < 2 or extrema_min.size < 2: # Not enough extrema
        print("Not enough extrema.")
        return np.copy(x), np.copy(x), np.zeros_like(x)

    def interp_factory(tx, ty): # Interpolates according to spline type
        if spline_type == "natural":
            return CubicSpline(tx, ty, bc_type="natural")
        if spline_type == "hermite":
            return PchipInterpolator(tx, ty)
        if spline_type == "akima":
            return Akima1DInterpolator(tx, ty)
        if spline_type == "linear":
            return interp1d(tx, ty, kind="linear", fill_value="extrapolate", assume_sorted=True)
        return CubicSpline(tx, ty, bc_type="natural")

    t_u, u = t[extrema_max], x[extrema_max] # Sample times and values at local maxima
    t_u_ext, u_ext = extend_signal(t_u, u, boundary, n_ext=max(2, min(4, len(u)-1))) # Boundary extension
    upper = interp_factory(t_u_ext, u_ext)(t) # Fit spline to extended maxima

    t_l, l = t[extrema_min], x[extrema_min] # Sample times and values at local minima
    t_l_ext, l_ext = extend_signal(t_l, l, boundary, n_ext=max(2, min(4, len(l)-1))) # Boundary extension
    lower = interp_factory(t_l_ext, l_ext)(t) # Fit spline to extended minima

    m = 1/2*(upper + lower) # mean envelope used for sifting
    return upper, lower, m

def custom_emd(t, x, max_imfs=10, max_sifts=50, sd_thresh=0.1, s_number=3, spline_type="natural", boundary="mirror", energy_tol=1e-3, range_tol=1e-3, min_res_extrema=2):
    r = x.copy().astype(float) # add working residual as float
    imfs = []
    for k in range(max_imfs):  # IMF loop
        # stop if residual is negligible (energy/range) or has too few extrema
        if np.std(r) < energy_tol*np.std(x) or np.ptp(r) < range_tol*np.ptp(x):
            break
        ds0 = np.diff(np.sign(np.where(np.diff(r)==0, 1e-12, np.diff(r))))
        n_ext0 = np.sum(ds0 < 0) + np.sum(ds0 > 0)
        if n_ext0 < min_res_extrema:
            break

        h = r.copy() # Proto IMF h from current residual
        consec_s_hits = 0 # s-number counter
        sd_prev = np.inf
        for s_it in range(max_sifts):
            upper, lower, m = make_envelopes(t, h, spline_type=spline_type, boundary=boundary) # build envelope for current h
            h_next = h-m # subtract mean envelope
            sd_val = sd_criterion(h, h_next)

            if is_imf(h_next):
                consec_s_hits += 1
            else:
                consec_s_hits = 0 # update counter

            h = h_next
            # SD + S-number stop or SD plateau stop
            if (sd_val < sd_thresh and consec_s_hits >= s_number) or (s_it > 5 and abs(sd_prev - sd_val) < 1e-5):
                break
            sd_prev = sd_val

        imfs.append(h)  # accept IMF
        r = r - h       # update residual
        # repeat robust stop on updated residual
        if np.std(r) < energy_tol*np.std(x) or np.ptp(r) < range_tol*np.ptp(x):
            break
        ds1 = np.diff(np.sign(np.where(np.diff(r)==0, 1e-12, np.diff(r))))
        n_ext1 = np.sum(ds1 < 0) + np.sum(ds1 > 0)
        if n_ext1 < min_res_extrema:
            break

    return imfs, r
            
            
def plot_imfs_stacked(t, imfs, residual=None, title="IMFs"):
    plt.figure(figsize=(10, 4.0 + 0.4*len(imfs)))
    off = 0.0
    for c in imfs:
        plt.plot(t, c + off)
        off += 1.2*np.nanmax(np.abs(c)+1e-12)
    if residual is not None:
        plt.plot(t, residual + off)
    plt.title(title); plt.xlabel("Time [s]"); plt.ylabel("Offset amp.")
    plt.show()

def plot_instant_freq(t, x, title):
    analytic = hilbert(x) # computes analytic signal using hilbert transform
    phase = np.unwrap(np.angle(analytic)) # extracts + unwraps instant phase to remove 2pi jumps
    f = np.gradient(phase, t)/(2*np.pi) # derivative

    # plot
    plt.figure(figsize=(10, 2.8))
    plt.plot(t, f)
    plt.title(title); plt.xlabel("Time [s]"); plt.ylabel("Frequency [Hz]")
    plt.show()
    
    return f # returns instant freq array

def plot_imfs_grid(t, imfs, residual=None, show_if=False, if_idx=1, fs=None):
    imfs = np.asarray(imfs)
    n_imf = imfs.shape[0]
    rows = n_imf + (1 if residual is not None else 0)
    if rows == 0:
        print("No IMFs to plot."); return
    fig, axes = plt.subplots(rows, 1, figsize=(10, max(6, 1.4*rows)), sharex=True)
    if rows == 1:
        axes = [axes]
    # IMFs
    for i in range(n_imf):
        axes[i].plot(t, imfs[i])
        axes[i].set_ylabel(f"IMF {i+1}")
    # Residual
    if residual is not None:
        axes[-1].plot(t, residual)
        axes[-1].set_ylabel("Residual")
        axes[-1].set_xlabel("Time [s]")
    else:
        axes[-1].set_xlabel("Time [s]")
    fig.suptitle("IMF + Hilbert Transform", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

    # Optional instantaneous frequency of one IMF
    if show_if and n_imf > 0:
        if fs is None:
            dt = np.median(np.diff(t))
            fs = 1.0/dt
        fig_if, axes_if = plt.subplots(n_imf, 1, figsize=(10, max(6, 1.2*n_imf)), sharex=True)
        if n_imf == 1:
            axes_if = [axes_if]
        for i in range(n_imf):
            z = hilbert(imfs[i])
            phase = np.unwrap(np.angle(z))
            f_inst = np.gradient(phase, t)/(2*np.pi)  # Hz
            axes_if[i].plot(t, f_inst)
            axes_if[i].set_ylabel(f"IF IMF {i+1} [Hz]")
        axes_if[-1].set_xlabel("Time [s]")
        fig_if.suptitle("Instantaneous Frequency of All IMFs", fontsize=14, y=1.02)
        plt.tight_layout(); plt.show()


def fft_imfs(t, imfs, residual=None, onesided=True, db=True, detrend=True, window='hann', pad_to=None, title="FFT of IMFs", fmax=60, crop_frac=0.05, use_psd=False):
    """
    FFT for each IMF (and optional residual). Returns (freqs_list, mags_list).
    - onesided: rFFT for real signals
    - db: plot 20*log10 magnitude
    - detrend: remove mean before FFT
    - window: None | 'hann' | array-like weights
    - pad_to: zero-pad length (int) or None
    """
    imfs = np.asarray(imfs)
    n_imf = imfs.shape[0]
    rows = n_imf + (1 if residual is not None else 0)
    if rows == 0:
        print("No IMFs."); return [], []

    # sampling
    dt = np.median(np.diff(t))
    fs = 1.0/dt

    def _prep(x):
        # crop edges to reduce end transients from EMD
        n = len(x); c = int(np.floor(crop_frac*n))
        if c > 0 and (2*c) < n:
            x = x[c:-c]
        x = x - np.mean(x) if detrend else x
        if isinstance(window, str):
            if window.lower() in ("hann","hanning"):
                w = np.hanning(len(x))
            else:
                raise ValueError("Unsupported window string")
            xw = x * w
            w_gain = np.sum(w)/len(w)   # coherent gain for amplitude spectra
        elif window is None:
            xw = x
            w_gain = 1.0
        else:
            w = np.asarray(window)
            if w.shape[0] != len(x):
                raise ValueError("window length must match signal")
            xw = x * w
            w_gain = np.sum(w)/len(w)
        return xw, w_gain

    def _fft_mag_simple(x, fs):
        N = len(x)
        Y = np.fft.fft(x)
        f = np.fft.fftfreq(N, d=1.0/fs)[:N//2]
        mag = 2.0/N * np.abs(Y[:N//2])
        return f, mag
    
    freqs_list, mags_list = [], []
    
    # figure
    fig, axes = plt.subplots(rows, 1, figsize=(10, max(6, 1.4*rows)), sharex=False)
    if rows == 1:
        axes = [axes]
    
    # IMFs
    for i in range(n_imf):
        x_in = imfs[i] - np.mean(imfs[i]) if detrend else imfs[i]
        f, mag = _fft_mag_simple(x_in, fs)
        freqs_list.append(f); mags_list.append(mag)
        axes[i].plot(f, mag)
        axes[i].set_ylabel(f"IMF {i+1}")
        axes[i].set_xlim(0, fmax)
    
    # Residual
    if residual is not None:
        x_in = residual - np.mean(residual) if detrend else residual
        f, mag = _fft_mag_simple(x_in, fs)
        freqs_list.append(f); mags_list.append(mag)
        axes[-1].plot(f, mag)
        axes[-1].set_ylabel("Residual")
        axes[-1].set_xlim(0, fmax)
        axes[-1].set_xlabel("Frequency [Hz]")
    else:
        axes[-1].set_xlabel("Frequency [Hz]")
    
    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return freqs_list, mags_list

def preset_from_two_freqs(f1, f2, A1=1.0, A2=1.0, fs=2000.0):
    # Sort tones
    f_low, f_high = sorted([float(f1), float(f2)])
    A_low, A_high = (float(A1), float(A2)) if f1 <= f2 else (float(A2), float(A1))

    # Ratio (separation hint)
    ratio = f_low / f_high if f_high > 0 else np.nan
    df = f_high - f_low
    fN = fs / 2.0

    # Core choice: attract high, avoid low
    f0 = 2.0 * f_high - f_low
    f0 = min(f0, 0.9 * fN)  # keep margin to Nyquist

    # Keep a gap from the attracted tone
    if abs(f0 - f_high) < 0.1 * max(f_high, 1e-12):
        f0 = min(1.2 * f_high, 0.9 * fN)

    # Conservative amplitude heuristic
    a0 = 0.8 * (A_high + A_low)
    phi = 0.0

    return {
        "a0": float(a0),
        "f0": float(f0),
        "phi": float(phi),
        "f_low": float(f_low),
        "f_high": float(f_high),
        "ratio": float(ratio),
        "df": float(df),
        "fs": float(fs)
    }

def make_double_mask_from_two_tones(t, f1, f2, A1=1.0, A2=1.0, fs=2000.0,
                                    hi_scale=0.8, lo_scale=0.4,
                                    phi_hi=0.0, phi_lo=np.pi/2):
    """
    Build two masking sinusoids (high-attraction + low-guard) and their sum.
    Based on the boundary-map and double-masking theory for close frequencies (e.g. 48–50 Hz).
    Returns:
      m_sum, m_hi, m_lo, params
    """

    # --- order tones ---
    f_low, f_high = sorted([float(f1), float(f2)])
    A_low, A_high = (float(A1), float(A2)) if f1 <= f2 else (float(A2), float(A1))
    fs = float(fs)
    fN = 0.5 * fs
    df = f_high - f_low

    # --- theoretical mask frequencies ---
    # High mask attracts upper tone but not the lower one
    f0_hi = f_high + df
    # Low mask prevents upward pull of lower tone
    f0_lo = max(f_low - df, 0.2)

    # keep within bounds
    f0_hi = min(f0_hi, 0.9 * fN)
    f0_lo = max(0.2, min(f0_lo, 0.9 * fN))

    # --- amplitudes ---
    # high mask: strong to dominate upper IMF attraction
    a0_hi = hi_scale * (A_high + A_low)
    # low mask: weaker to act as guard
    a0_lo = lo_scale * (A_high + A_low)

    # --- build signals ---
    m_hi = a0_hi * np.sin(2 * np.pi * f0_hi * t + phi_hi)
    m_lo = a0_lo * np.sin(2 * np.pi * f0_lo * t + phi_lo)
    m_sum = m_hi + m_lo

    params = {
        "context": {
            "f_low": f_low,
            "f_high": f_high,
            "df": df,
            "fs": fs,
            "ratio": (f_low / f_high) if f_high > 0 else np.nan
        },
        "mask_high": {"a0": a0_hi, "f0": f0_hi, "phi": phi_hi},
        "mask_low":  {"a0": a0_lo, "f0": f0_lo, "phi": phi_lo}
    }
    return m_sum, m_hi, m_lo, params


In [3]:
fif_path = "Lectures/TTK7/signal.fif"
sig_name = "fif_ch0"

x_fif, t_fif, fs_fif = None, None, None
# 1) Try Raw
try:
    raw = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)
    x_fif = raw.get_data(picks=[0])[0]
    fs_fif = float(raw.info["sfreq"])
    t_fif = np.arange(x_fif.size, dtype=float) / fs_fif
    src_type = "Raw"
except Exception:
    # 2) Try Epochs
    try:
        epochs = mne.read_epochs(fif_path, preload=True, verbose=False)
        fs_fif = float(epochs.info["sfreq"])
        x_fif = epochs.get_data(picks=[0]).mean(axis=0)
        t_fif = np.arange(x_fif.size, dtype=float) / fs_fif
        src_type = "Epochs"
    except Exception:
        # 3) Try Evoked
        evk = mne.read_evokeds(fif_path, condition=0, verbose=False)
        evk = evk[0] if isinstance(evk, list) else evk
        x_fif = evk.data[0]
        fs_fif = float(evk.info["sfreq"])
        t_fif = np.asarray(evk.times, dtype=float)
        src_type = "Evoked"

print(f"Loaded {src_type}: {len(x_fif)} samples @ {fs_fif:.2f} Hz")

# --- Generate variants ---
np.random.seed(0)
x_clean = x_fif
x_noise = add_noise_to_signal(x_clean, snr_db=10.0)
x_noff  = add_offset_to_signal(x_noise, offset=2.0)

SIGNALS = SIGNALS if "SIGNALS" in globals() else {}
SIGNALS[f"{sig_name}_clean"] = {"t": t_fif, "x": x_clean}
SIGNALS[f"{sig_name}_noise"] = {"t": t_fif, "x": x_noise}
SIGNALS[f"{sig_name}_noise_and_offset"] = {"t": t_fif, "x": x_noff}

# expose fs/t if not already set
fs = globals().get("fs", fs_fif)
t  = globals().get("t", t_fif)

# ---- Shared storage for signals from block 1 ----
try:
    SIGNALS
except NameError:
    SIGNALS = {}
try:
    t
except NameError:
    t = None

# Ensure fs/duration exist
try:
    fs
except NameError:
    fs = 2000
try:
    duration
except NameError:
    duration = 2.0

# Ensure helpers exist or define them here
# add_noise_to_signal, add_offset_to_signal

# Always add/update signals
t_default = np.arange(0, duration, 1.0/fs)
x_default = make_signal_from_freq(duration, amps, freqs, t_default)
np.random.seed(0)
x_noise = add_noise_to_signal(x_default, 10)
x_noise_offset = add_offset_to_signal(x_noise, 2)

SIGNALS.setdefault("synthetic_clean", {"t": t_default, "x": x_default})
SIGNALS.setdefault("synthetic_noise", {"t": t_default, "x": x_noise})
SIGNALS.setdefault("synthetic_noise_and_offset", {"t": t_default, "x": x_noise_offset})

# Utility: get selected signal
def get_signal(name):
    entry = SIGNALS.get(name)
    if entry is None:
        entry = SIGNALS[next(iter(SIGNALS))]

    xx = np.asarray(entry["x"], dtype=float).squeeze()
    tt = entry.get("t", None)
    if tt is None:
        tt = np.arange(xx.size, dtype=float)
    else:
        tt = np.asarray(tt, dtype=float).squeeze()
        if tt.size != xx.size:
            tt = np.arange(xx.size, dtype=float)  # fallback to index time

    return tt, xx

Loaded Epochs: 1 samples @ 250.00 Hz


/tmp/ipykernel_1138356/3158732458.py:7: RuntimeWarning: This filename (Lectures/TTK7/signal.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)
/tmp/ipykernel_1138356/3158732458.py:15: RuntimeWarning: This filename (Lectures/TTK7/signal.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(fif_path, preload=True, verbose=False)


In [4]:
# Widgets
w_add_noise = Checkbox(value=False, description="Add Noise")
w_snr = FloatSlider(value=20.0, min=0.0, max=60.0, step=1.0, description="SNR [dB]")
w_add_offset = Checkbox(value=False, description="Add Offset")
w_offset = FloatSlider(value=0.5, min=-5.0, max=5.0, step=0.05, description="Offset")

def _toggle_noise(change=None):
    w_snr.disabled  = not w_add_noise.value
def _toggle_offset(change=None):
    w_offset.disabled = not w_add_offset.value

_toggle_noise(); _toggle_offset()
w_add_noise.observe(_toggle_noise, names='value')
w_add_offset.observe(_toggle_offset, names='value')

def run(add_noise, snr_db, add_offset, offset):
    x = make_signal_from_freq(t, amps, freqs, t)
    if add_noise:
        x = add_noise_to_signal(x, snr_db)
    if add_offset:
        x = add_offset_to_signal(x, offset)

    plt.figure(figsize=(10, 2.8))
    plt.plot(t, x)
    plt.xlim(0, duration)
    plt.title(f"Signal | noise={add_noise} (SNR={snr_db:.0f} dB) | offset={add_offset} (Δ={offset:.2f})")
    plt.xlabel("Time [s]"); plt.ylabel("Amplitude")
    plt.show()

ui = VBox([
    HBox([w_add_noise, w_snr]),
    HBox([w_add_offset, w_offset]),
])
out = interactive_output(run, {
    'add_noise':  w_add_noise,
    'snr_db':     w_snr,
    'add_offset': w_add_offset,
    'offset':     w_offset,
})
display(ui, out)

Output()

In [5]:
# Widgets — only EMD parameters + which signal to use
w_sig = Dropdown(options=sorted(SIGNALS.keys()), value=sorted(SIGNALS.keys())[0], description="Signal")
w_spline = Dropdown(options=[("Natural cubic","natural"), ("Hermite (PCHIP)","hermite"),
                             ("Akima","akima"), ("Linear","linear")],
                    value="natural", description="Envelope")
w_boundary = Dropdown(options=[("Mirror","mirror"), ("Mirror+taper","mirror_taper"),
                               ("Linear","linear"), ("None","none")],
                      value="mirror", description="Boundary")
w_max_imfs = IntSlider(value=6, min=1, max=12, step=1, description="Max IMFs")
w_max_sifts = IntSlider(value=20, min=1, max=200, step=1, description="Max sifts")
w_sd = FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description="SD thresh")
w_snum = IntSlider(value=3, min=1, max=10, step=1, description="S-number")

btn = Button(description="Run custom EMD")
out = Output()

# Refresh dropdown AFTER updates
opts = sorted(SIGNALS.keys())
w_sig.options = opts
if w_sig.value not in opts:
    w_sig.value = opts[0]

def _run(_=None):
    out.clear_output(wait=True)
    with out:
        tt, xx = get_signal(w_sig.value)

        plot_signal(tt, xx, duration, f"Input signal: {w_sig.value}")

        imfs, r = custom_emd(
            tt, xx,
            max_imfs=w_max_imfs.value,
            max_sifts=w_max_sifts.value,
            sd_thresh=w_sd.value,
            s_number=w_snum.value,
            spline_type=w_spline.value,
            boundary=w_boundary.value
        )
        if not imfs:
            print("No IMFs extracted.")
            return
        plot_imfs_stacked(tt, imfs, residual=r, title=f"Custom EMD — {w_sig.value}")
        plot_imfs_grid(tt, imfs, residual=r, show_if=True, if_idx=1, fs=1/np.median(np.diff(tt)))
        freqs, mags = fft_imfs(tt, imfs, residual=r, onesided=True, db=True)

btn.on_click(_run)

display(VBox([
    HBox([w_sig]), # Selects signal
    HBox([w_spline, w_boundary]), # Selects which spline and boundary to use
    HBox([w_max_imfs, w_max_sifts]), # Upper bound on imfs (low -> leaves energy in residual, high -> risk splitting modes or extracting noise).
    # max_sifts -> upper bound on sifts -> too low, mean not near zero (under-sifted), too high -> cleaner IMF, longer runtime, possible over-sifting
    HBox([w_sd, w_snum]), # SD threshold -> convergence test. Lower -> more iterations, cleaner symmetry -> risk mode splitting. Higher -> faster stop, rougher IMFS
    # IMF validity for S consecutive sift. Large -> more conservative, cleaner modes, longer time. Smaller -> faster, but more residual
    btn,
    out
]))


In [6]:
# block 4 — PyEMD with widgets (EMD-only params)
# Expects: SIGNALS dict from block 1. Uses PyEMD for decomposition.

# Widgets — PyEMD-exposed EMD params and signal selector
w_sig = Dropdown(options=sorted(SIGNALS.keys()), value=sorted(SIGNALS.keys())[0], description="Signal")
w_spline_kind = Dropdown(
    options=[
        ("Cubic", "cubic"),
        ("Linear", "slinear"),
        ("Natural", "natural"),
        ("Hermite (PCHIP)", "hermite"),
    ],
    value="cubic",
    description="Spline"
)
w_nbsym = IntSlider(value=2, min=0, max=6, step=1, description="nbsym")
w_max_imfs = IntSlider(value=6, min=1, max=12, step=1, description="Max IMFs")
w_max_sifts = IntSlider(value=50, min=1, max=500, step=1, description="Max sifts")

btn = Button(description="Run EMD")
out = Output()

def _run(_=None):
    out.clear_output(wait=True)
    with out:
        tt, xx = get_signal(w_sig.value)

        plot_signal(tt, xx, duration, f"Input signal: {w_sig.value}")

        if w_spline_kind.value in ("natural", "hermite"):
            # Use homemade EMD to support 'natural' and 'hermite' envelope splines
            imfs, r = custom_emd(
                tt, xx,
                max_imfs=int(w_max_imfs.value),
                max_sifts=int(w_max_sifts.value),
                sd_thresh=0.1,          # default since PyEMD UI doesn’t expose these
                s_number=3,
                spline_type=w_spline_kind.value,
                boundary="mirror"
            )
        else:
            # Use PyEMD for built-in spline kinds
            emd = EMD(spline_kind=w_spline_kind.value, nbsym=int(w_nbsym.value))
            if hasattr(emd, "MAX_ITERATIONS"):
                try:
                    emd.MAX_ITERATIONS = int(w_max_sifts.value)
                except Exception:
                    pass
            try:
                c = emd.emd(xx, tt, max_imf=int(w_max_imfs.value))
            except TypeError:
                c = emd.emd(xx, tt)
            if c is None or (hasattr(c, "ndim") and c.ndim == 0):
                print("No IMFs extracted.")
                return
            imfs = [c_i for c_i in c[:int(w_max_imfs.value)]] if getattr(c, "ndim", 1) == 2 else [c]
            r = xx - np.sum(np.array(imfs), axis=0)
        label_map = {
            "cubic": "Cubic",
            "slinear": "Linear",
            "natural": "Natural",
            "hermite": "Hermite (PCHIP)"
        }
        title_label = label_map.get(w_spline_kind.value, w_spline_kind.value)

        plot_imfs_stacked(tt, imfs, residual=r, title=f"EMD ({title_label}) — {w_sig.value}")
        fs = 1.0 / np.median(np.diff(tt))
        plot_imfs_grid(tt, imfs, residual=r, show_if=True, fs=fs)
        freqs, mags = fft_imfs(t, imfs, residual=r, onesided=True, db=True)

btn.on_click(_run)

display(VBox([
    HBox([w_sig]),
    HBox([w_spline_kind, w_nbsym]),
    HBox([w_max_imfs, w_max_sifts]),
    btn,
    out
]))


In [7]:
# --- Minimal UI to compute and show parameters ---
w_f1 = FloatSlider(value=48.0, min=0.1, max=500.0, step=0.1, description="f1 [Hz]")
w_A1 = FloatSlider(value=1.0,  min=0.0, max=10.0, step=0.05, description="A1")
w_f2 = FloatSlider(value=50.0, min=0.1, max=500.0, step=0.1, description="f2 [Hz]")
w_A2 = FloatSlider(value=1.0,  min=0.0, max=10.0, step=0.05, description="A2")
w_fs = FloatSlider(value=2000.0, min=50.0, max=10000.0, step=10.0, description="fs [Hz]")

btn = Button(description="Compute preset")
out = Output()

def on_compute(_=None):
    out.clear_output(wait=True)
    params = preset_from_two_freqs(w_f1.value, w_f2.value, w_A1.value, w_A2.value, w_fs.value)
    with out:
        print("Preset parameters")
        print(f"  a0  = {params['a0']:.6g}")
        print(f"  f0  = {params['f0']:.6g} Hz")
        print(f"  phi = {params['phi']:.6g} rad")
        print("\nContext")
        print(f"  f_low  = {params['f_low']:.6g} Hz")
        print(f"  f_high = {params['f_high']:.6g} Hz")
        print(f"  ratio  = {params['ratio']:.6g}")
        print(f"  df     = {params['df']:.6g} Hz")
        print(f"  fs     = {params['fs']:.6g} Hz")

btn.on_click(on_compute)

display(VBox([
    HBox([w_f1, w_A1]),
    HBox([w_f2, w_A2]),
    HBox([w_fs]),
    btn,
    out
]))

In [8]:
def mask(t, a0, f0, phi):
    return a0*np.sin(2*np.pi*f0*t + phi)

# Widgets — identical to custom EMD + manual mask params
w_sig = Dropdown(options=sorted(SIGNALS.keys()), value=sorted(SIGNALS.keys())[0], description="Signal")
w_spline = Dropdown(options=[("Natural cubic","natural"), ("Hermite (PCHIP)","hermite"),
                             ("Akima","akima"), ("Linear","linear")],
                    value="natural", description="Envelope")
w_boundary = Dropdown(options=[("Mirror","mirror"), ("Mirror+taper","mirror_taper"),
                               ("Linear","linear"), ("None","none")],
                      value="mirror", description="Boundary")
w_max_imfs = IntSlider(value=6, min=1, max=12, step=1, description="Max IMFs")
w_max_sifts = IntSlider(value=20, min=1, max=200, step=1, description="Max sifts")
w_sd = FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description="SD thresh")
w_snum = IntSlider(value=3, min=1, max=10, step=1, description="S-number")

# Manual mask parameters
w_a0  = FloatSlider(value=1.6, min=0.0, max=10.0, step=0.05, description="a0")
w_f0  = FloatSlider(value=52.0, min=0.1, max=500.0, step=0.1, description="f0 [Hz]")
w_phi = FloatSlider(value=0.0, min=0.0, max=2*np.pi, step=0.05, description="phi [rad]")

btn = Button(description="Run masked EMD")
out = Output()

def run(_=None):
    out.clear_output(wait=True)
    with out:
        t, x = get_signal(w_sig.value)
        fs = 1.0/np.median(np.diff(t))
        m = mask(t, w_a0.value, w_f0.value, w_phi.value)

        # EMD on x + mask
        imfs_masked, _ = custom_emd(
            t, x + m,
            max_imfs=int(w_max_imfs.value),
            max_sifts=int(w_max_sifts.value),
            sd_thresh=float(w_sd.value),
            s_number=int(w_snum.value),
            spline_type=w_spline.value,
            boundary=w_boundary.value
        )
        if not imfs_masked:
            print("No IMFs extracted."); return

        # Remove known mask from IMF1
        imfs = [c.copy() for c in imfs_masked]
        imfs[0] = imfs[0] - m
        r = x - np.sum(np.asarray(imfs), axis=0)

        plot_signal(t, x, t[-1]-t[0]+(t[1]-t[0]), f"Input: {w_sig.value}")
        plt.figure(figsize=(10,2.2)); plt.plot(t, m); plt.title(f"Mask (a0={w_a0.value:.2f}, f0={w_f0.value:.2f} Hz)"); plt.xlabel("Time [s]"); plt.show()
        plot_imfs_stacked(t, imfs, residual=r, title=f"Masked custom EMD — {w_sig.value}")
        plot_imfs_grid(t, imfs, residual=r, show_if=True, fs=fs)
        fft_imfs(t, imfs, residual=r, onesided=True, db=True, title="FFT of Masked IMFs", fmax=min(100, 0.5*fs))

btn.on_click(run)

display(VBox([
    HBox([w_sig]),
    HBox([w_spline, w_boundary]),
    HBox([w_max_imfs, w_max_sifts]),
    HBox([w_sd, w_snum]),
    HBox([w_a0, w_f0, w_phi]),
    btn,
    out
]))

In [9]:
# --- Double-mask sinus + custom EMD (manual parameters) ---
# EMD parameters (same as before)
w_sig = Dropdown(options=sorted(SIGNALS.keys()), value=sorted(SIGNALS.keys())[0], description="Signal")
w_spline = Dropdown(options=[("Natural cubic","natural"), ("Hermite (PCHIP)","hermite"),
                             ("Akima","akima"), ("Linear","linear")],
                    value="natural", description="Envelope")
w_boundary = Dropdown(options=[("Mirror","mirror"), ("Mirror+taper","mirror_taper"),
                               ("Linear","linear"), ("None","none")],
                      value="mirror", description="Boundary")
w_max_imfs  = IntSlider(value=6,  min=1, max=12,  step=1, description="Max IMFs")
w_max_sifts = IntSlider(value=20, min=1, max=200, step=1, description="Max sifts")
w_sd   = FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description="SD thresh")
w_snum = IntSlider(value=3,   min=1, max=10,  step=1, description="S-number")

# --- Manual mask parameters (preset values can be applied separately) ---
w_a0_hi  = FloatSlider(value=1.0, min=0.0, max=5.0, step=0.05, description="a0_hi")
w_f0_hi  = FloatSlider(value=55.0, min=0.1, max=500.0, step=0.1, description="f0_hi [Hz]")
w_phi_hi = FloatSlider(value=0.0, min=0.0, max=2*np.pi, step=0.05, description="phi_hi [rad]")

w_a0_lo  = FloatSlider(value=0.5, min=0.0, max=5.0, step=0.05, description="a0_lo")
w_f0_lo  = FloatSlider(value=40.0, min=0.1, max=500.0, step=0.1, description="f0_lo [Hz]")
w_phi_lo = FloatSlider(value=np.pi/2, min=0.0, max=2*np.pi, step=0.05, description="phi_lo [rad]")

btn = Button(description="Run double-masked EMD")
out = Output()

def run(_=None):
    out.clear_output(wait=True)
    with out:
        t, x = get_signal(w_sig.value)
        fs = 1.0 / np.median(np.diff(t))

        # Construct the two masks manually
        m_hi = w_a0_hi.value * np.sin(2*np.pi*w_f0_hi.value*t + w_phi_hi.value)
        m_lo = w_a0_lo.value * np.sin(2*np.pi*w_f0_lo.value*t + w_phi_lo.value)
        m_sum = m_hi + m_lo

        # Run EMD on x + total mask
        imfs_masked, _ = custom_emd(
            t, x + m_sum,
            max_imfs=int(w_max_imfs.value),
            max_sifts=int(w_max_sifts.value),
            sd_thresh=float(w_sd.value),
            s_number=int(w_snum.value),
            spline_type=w_spline.value,
            boundary=w_boundary.value
        )
        if not imfs_masked:
            print("No IMFs extracted.")
            return

        # Remove known masks from IMF1
        imfs = [c.copy() for c in imfs_masked]
        imfs[0] = imfs[0] - m_sum
        r = x - np.sum(np.asarray(imfs), axis=0)

        # Plot signal and masks
        plot_signal(t, x, t[-1]-t[0]+(t[1]-t[0]), f"Input: {w_sig.value}")
        plt.figure(figsize=(10,2.4))
        plt.plot(t, m_hi, label=f"High mask f0={w_f0_hi.value:.2f} Hz")
        plt.plot(t, m_lo, label=f"Low mask f0={w_f0_lo.value:.2f} Hz", alpha=0.8)
        plt.plot(t, m_sum, label="Sum", alpha=0.6)
        plt.legend(); plt.title("Manual double masks"); plt.xlabel("Time [s]"); plt.show()

        # Plot IMFs
        plot_imfs_stacked(t, imfs, residual=r, title=f"Double-masked EMD — {w_sig.value}")
        plot_imfs_grid(t, imfs, residual=r, show_if=True, fs=fs)
        fft_imfs(t, imfs, residual=r, onesided=True, db=True,
                 title="FFT of Masked IMFs", fmax=min(100, 0.5*fs))

btn.on_click(run)

display(VBox([
    HBox([w_sig]),
    HBox([w_spline, w_boundary]),
    HBox([w_max_imfs, w_max_sifts]),
    HBox([w_sd, w_snum]),
    HTML("<b>Mask 1 (High):</b>"),
    HBox([w_a0_hi, w_f0_hi, w_phi_hi]),
    HTML("<b>Mask 2 (Low):</b>"),
    HBox([w_a0_lo, w_f0_lo, w_phi_lo]),
    btn,
    out
]))


In [10]:
# Signal selector
w_sig = Dropdown(options=sorted(SIGNALS.keys()), value=sorted(SIGNALS.keys())[0], description="Signal")

# Inner EMD settings
w_spline   = Dropdown(options=[("Cubic","cubic"), ("Linear","slinear")], value="cubic", description="Spline")
w_nbsym    = IntSlider(value=2, min=0, max=6, step=1, description="nbsym")
w_max_imfs = IntSlider(value=6, min=1, max=16, step=1, description="Max IMFs")
w_max_sifts= IntSlider(value=50, min=1, max=500, step=1, description="Max sifts")

# EEMD settings
w_trials   = IntSlider(value=50, min=5, max=500, step=5, description="Ensemble")
w_noise    = FloatSlider(value=0.2, min=0.0, max=2.0, step=0.01, description="Noise width")

btn = Button(description="Run EEMD")
out = Output()

def run(_=None):
    out.clear_output(wait=True)
    with out:
        t, x = get_signal(w_sig.value)
        fs = 1.0/np.median(np.diff(t))

        # Configure inner EMD used by EEMD
        emd = EMD(spline_kind=w_spline.value, nbsym=int(w_nbsym.value))
        if hasattr(emd, "MAX_ITERATIONS"):
            try:
                emd.MAX_ITERATIONS = int(w_max_sifts.value)
            except Exception:
                pass

        # Configure EEMD
        eemd = EEMD()
        eemd.trials = int(w_trials.value)
        # noise_width is std of added white noise scaled to data std
        eemd.noise_width = float(w_noise.value)
        eemd.EMD = emd

        # Decompose
        try:
            C = eemd.eemd(x, t, max_imf=int(w_max_imfs.value))
        except TypeError:
            C = eemd.eemd(x, t)
        if C is None or (hasattr(C, "ndim") and C.ndim == 0):
            print("No IMFs extracted."); return

        imfs = [C[i] for i in range(min(C.shape[0], int(w_max_imfs.value)))] if C.ndim == 2 else [C]
        r = x - np.sum(np.asarray(imfs), axis=0)

        # Plots
        plot_signal(t, x, t[-1]-t[0]+(t[1]-t[0]), f"Input: {w_sig.value}")
        plot_imfs_stacked(t, imfs, residual=r, title=f"EEMD ({w_spline.label}, trials={w_trials.value}, noise={w_noise.value})")
        plot_imfs_grid(t, imfs, residual=r, show_if=True, fs=fs)
        fft_imfs(t, imfs, residual=r, onesided=True, db=True, title="FFT of EEMD IMFs", fmax=min(100, 0.5*fs))

btn.on_click(run)

display(VBox([
    HBox([w_sig]),
    HBox([w_spline, w_nbsym]),
    HBox([w_max_imfs, w_max_sifts]),
    HBox([w_trials, w_noise]),
    btn,
    out
]))

In [11]:
# ---- Signal selector (expects SIGNALS + plotting helpers already defined) ----
w_sig = Dropdown(options=sorted(SIGNALS.keys()), value=sorted(SIGNALS.keys())[0], description="Signal")

# ---- Inner EMD settings ----
w_spline    = Dropdown(options=[("Cubic","cubic"), ("Linear","slinear")], value="cubic", description="Spline")
w_nbsym     = IntSlider(value=4, min=0, max=6, step=1, description="nbsym")
w_max_imfs  = IntSlider(value=6, min=1, max=16, step=1, description="Max IMFs")
w_max_sifts = IntSlider(value=200, min=10, max=1000, step=10, description="Max sifts")

# ---- CEEMDAN settings ----
w_trials    = IntSlider(value=300, min=20, max=1000, step=20, description="Ensemble")
w_noise     = FloatSlider(value=0.5, min=0.0, max=2.0, step=0.01, description="Noise width")

# ---- Optional mirror padding to reduce end effects ----
w_pad_on    = Checkbox(value=True, description="Mirror pad")
w_pad_sec   = FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, description="Pad seconds")

btn = Button(description="Run CEEMDAN")
out = Output()

def _mirror_pad(t, x, pad_sec):
    # Determine sampling step from median dt
    dt = float(np.median(np.diff(t)))
    fs = 1.0/dt
    n_pad = int(round(pad_sec*fs))
    if n_pad <= 0:
        return t, x, 0
    # Reflect signal at both ends
    x_pad = np.pad(x, (n_pad, n_pad), mode='reflect')
    # Build extended time vector
    t0, t1 = t[0], t[-1]
    t_left  = t0 - np.arange(n_pad, 0, -1)*dt
    t_right = t1 + np.arange(1, n_pad+1)*dt
    t_pad = np.concatenate([t_left, t, t_right])
    return t_pad, x_pad, n_pad

def run_ceemdan(_=None):
    out.clear_output(wait=True)
    with out:
        # Load signal
        def get_signal(name):
            entry = SIGNALS.get(name, next(iter(SIGNALS.values())))
            tt = entry.get("t", None)
            xx = entry["x"]
            if tt is None:
                tt = np.arange(len(xx), dtype=float)
            return tt, xx

        t, x = get_signal(w_sig.value)
        dt = float(np.median(np.diff(t)))
        fs = 1.0/dt

        # Optional mirror padding
        if w_pad_on.value and w_pad_sec.value > 0.0:
            t_in, x_in, n_pad = _mirror_pad(t, x, w_pad_sec.value)
        else:
            t_in, x_in, n_pad = t, x, 0

        # Configure inner EMD
        emd = EMD(spline_kind=w_spline.value, nbsym=int(w_nbsym.value))
        # Max sifts, if available in this PyEMD version
        if hasattr(emd, "MAX_ITERATIONS"):
            try:
                emd.MAX_ITERATIONS = int(w_max_sifts.value)
            except Exception:
                pass

        # Configure CEEMDAN
        ce = CEEMDAN()
        ce.trials = int(w_trials.value)
        ce.noise_width = float(w_noise.value)
        ce.ext_EMD = emd

        # Decompose
        try:
            C = ce.ceemdan(x_in, t_in, max_imf=int(w_max_imfs.value))
        except TypeError:
            C = ce.ceemdan(x_in, t_in)

        if C is None or (hasattr(C, "ndim") and C.ndim < 1):
            print("No IMFs extracted."); return

        # Crop padding
        if n_pad > 0:
            C = C[:, n_pad:-n_pad]

        # Enforce max_imfs UI limit
        if C.ndim == 2 and C.shape[0] > int(w_max_imfs.value):
            C = C[:int(w_max_imfs.value), :]

        imfs = [C[i] for i in range(C.shape[0])] if C.ndim == 2 else [C]
        r = x - np.sum(np.asarray(imfs), axis=0)

        # Plots (helpers must exist: plot_signal, plot_imfs_stacked, plot_imfs_grid, fft_imfs)
        plot_signal(t, x, t[-1]-t[0]+(t[1]-t[0]), f"Input: {w_sig.value}")
        plot_imfs_stacked(t, imfs, residual=r,
                          title=f"CEEMDAN ({w_spline.label}, trials={w_trials.value}, noise={w_noise.value}, pad={w_pad_sec.value if w_pad_on.value else 0}s)")
        plot_imfs_grid(t, imfs, residual=r, show_if=True, fs=fs)
        fft_imfs(t, imfs, residual=r, onesided=True, db=True,
                 title="FFT of CEEMDAN IMFs", fmax=min(100, 0.5*fs))

btn.on_click(run_ceemdan)

display(VBox([
    HBox([w_sig]),
    HBox([w_spline, w_nbsym]),
    HBox([w_max_imfs, w_max_sifts]),
    HBox([w_trials, w_noise]),
    HBox([w_pad_on, w_pad_sec]),
    btn,
    out
]))


In [12]:
# --- VMD block ---
from vmdpy import VMD

# expects: SIGNALS, plot_signal, plot_imfs_stacked, plot_imfs_grid, fft_imfs

w_sig    = Dropdown(options=sorted(SIGNALS.keys()), value=sorted(SIGNALS.keys())[0], description="Signal")
w_K      = IntSlider(value=2, min=1, max=8, step=1, description="K")
w_alpha  = FloatSlider(value=2000.0, min=10.0, max=20000.0, step=10.0, description="alpha")
w_tau    = FloatSlider(value=0.0, min=0.0, max=0.5, step=0.01, description="tau")
w_DC     = Checkbox(value=False, description="DC mode")
w_init   = Dropdown(options=[("uniform",1),("random",0)], value=1, description="init")
w_tol    = FloatSlider(value=1e-7, min=1e-9, max=1e-3, step=1e-9, readout_format=".1e", description="tol")
btn_vmd  = Button(description="Run VMD")
out_vmd  = Output()

def run_vmd(_=None):
    out_vmd.clear_output(wait=True)
    with out_vmd:
        t, x = get_signal(w_sig.value)
        dt = float(np.median(np.diff(t))); fs = 1.0/dt
        u, u_hat, omega = VMD(x, alpha=w_alpha.value, tau=w_tau.value, K=w_K.value,
                              DC=int(w_DC.value), init=w_init.value, tol=w_tol.value)
        imfs = [u[k] for k in range(u.shape[0])]
        r = x - np.sum(u, axis=0)
        plot_signal(t, x, t[-1]-t[0]+(t[1]-t[0]), f"Input: {w_sig.value}")
        plot_imfs_stacked(t, imfs, residual=r,
                          title=f"VMD (K={w_K.value}, alpha={w_alpha.value:.0f}, tau={w_tau.value}, DC={w_DC.value})")
        plot_imfs_grid(t, imfs, residual=r, show_if=True, fs=fs)
        fft_imfs(t, imfs, residual=r, onesided=True, db=True, title="FFT of VMD modes", fmax=min(100, 0.5*fs))

btn_vmd.on_click(run_vmd)
display(VBox([HBox([w_sig]), HBox([w_K, w_alpha]), HBox([w_tau, w_DC, w_init]), HBox([w_tol]), btn_vmd, out_vmd]))
